# 12-Qubit Google Supremacy Circuit: Tetron MBQC vs Direct Gate

What this notebook does:

1. **Load** one of Google's 12-qubit random-circuit-sampling files
   (`circuit_n12_m14_s0_e0_pEFGH.py`).
2. **Translate** every Cirq operation into a 24-site tetron MBQC circuit on an
   8x3 layout, using `qubit_mapping.py` and `mbqc_translated_gates.py`.
3. **Build a direct 12-qubit Qiskit reference** that applies the same
   `sqrt(X)`, `sqrt(Y)`, `sqrt(W)`, `fSim(pi/2, pi/6)` gates as ideal unitaries.
4. **Sample** both circuits noiselessly via `SamplerV2` (`shots = 4000`).
5. **Plot** the output bitstring distributions side-by-side.

Conventions used:

- `QUBIT_ORDER` in the Google file is `[(3,3), (3,4), ..., (5,6)]`. We treat
  the leftmost character of a bitstring as qubit (3,3) (i.e. **Google order**).
  Qiskit's `get_counts()` returns bitstrings in the opposite order; we reverse
  them.
- Every two-qubit gate becomes `fSim(theta=pi/2, phi=pi/6)`. The actual Cirq
  angles in the file vary slightly around those values, but the tetron MBQC
  two-qubit gadget is calibrated at this fixed pair.
- The per-edge `Rz(...)` wrappers in the Cirq file are **dropped**, in both
  circuits, so the unitaries being compared are identical.


## 1. Imports and configuration

In [2]:
import os, sys, importlib.util
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import cirq

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.compiler import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler

# This notebook lives at the repo root. Make the tetron helpers importable.
REPO_ROOT  = os.path.abspath('.')
TETRON_DIR = os.path.join(REPO_ROOT, 'src', 'tetron')
GOOGLE_DIR = os.path.join(REPO_ROOT, 'google_supremacy_circuit_files')

if TETRON_DIR not in sys.path:
    sys.path.insert(0, TETRON_DIR)

from qubit_mapping import (
    grid_to_qiskit_index,
    grid_to_sq_ancilla_index,
    grid_edge_to_qiskit_indices,
    GRID_TO_LOGICAL,
)
from mbqc_translated_gates import MBQCTranslatedGates, fSim_matrix, sqrt_W_matrix

print('Imports OK.  Tetron dir:', TETRON_DIR)


Imports OK.  Tetron dir: e:\git_repo\mbqc-circuit-comparison\src\tetron


## 2. Load a Google supremacy circuit

In [3]:
CIRCUIT_FILE = os.path.join(
    GOOGLE_DIR, 'circuit_n12_m14_s0_e0_pEFGH.py'
)

def load_cirq_circuit(path):
    """exec() a Google circuit data file and return (QUBIT_ORDER, CIRCUIT)."""
    spec = importlib.util.spec_from_file_location('google_circuit', path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.QUBIT_ORDER, mod.CIRCUIT

QUBIT_ORDER, CIRCUIT = load_cirq_circuit(CIRCUIT_FILE)
print(f'Loaded {os.path.basename(CIRCUIT_FILE)}')
print(f'  qubits  = {len(QUBIT_ORDER)}')
print(f'  moments = {len(CIRCUIT)}')


Loaded circuit_n12_m14_s0_e0_pEFGH.py
  qubits  = 12
  moments = 57


## 3. Translate the Cirq circuit into a tetron MBQC circuit

Strategy:

- Allocate 24 qubits (T1..T24 mapped to Qiskit indices 0..23) and one large
  classical register `c_int` to hold every MBQC intermediate measurement.
- Walk every Cirq operation:
  - `(X**0.5)`               -> `gate_sqrt_X` on (data, ancilla).
  - `(Y**0.5)`               -> `gate_sqrt_Y`.
  - `PhasedXPowGate(0.25, 0.5)` -> `gate_sqrt_W`.
  - `Rz(...)`                -> dropped.
  - `FSimGate(...)`          -> `gate_two_qubit` at `(pi/2, pi/6)`.
- A 12-bit `c_out` register holds the final data-qubit measurements in the
  same order as `QUBIT_ORDER`.


In [4]:
FSIM_THETA = np.pi / 2
FSIM_PHI   = np.pi / 6


def _is_sqrt_X(g):
    return isinstance(g, cirq.XPowGate) and np.isclose(g.exponent, 0.5)

def _is_sqrt_Y(g):
    return isinstance(g, cirq.YPowGate) and np.isclose(g.exponent, 0.5)

def _is_sqrt_W(g):
    return (isinstance(g, cirq.PhasedXPowGate)
            and np.isclose(g.phase_exponent, 0.25)
            and np.isclose(g.exponent, 0.5))


def count_classical_bits(cirq_circuit,
                         fsim_theta=FSIM_THETA, fsim_phi=FSIM_PHI):
    """Total intermediate-measurement bits the MBQC translation needs."""
    fsim_bits = MBQCTranslatedGates.n_bits_two_qubit(fsim_theta, fsim_phi)
    n = 0
    for moment in cirq_circuit:
        for op in moment.operations:
            g = op.gate
            if   _is_sqrt_X(g): n += MBQCTranslatedGates.N_BITS['sqrt_X']
            elif _is_sqrt_Y(g): n += MBQCTranslatedGates.N_BITS['sqrt_Y']
            elif _is_sqrt_W(g): n += MBQCTranslatedGates.N_BITS['sqrt_W']
            elif isinstance(g, cirq.ZPowGate):  pass         # dropped
            elif isinstance(g, cirq.FSimGate):  n += fsim_bits
    return n


def build_mbqc_tetron_circuit(qubit_order, cirq_circuit,
                              fsim_theta=FSIM_THETA, fsim_phi=FSIM_PHI):
    """24-qubit Qiskit MBQC translation of the Cirq circuit."""
    n_int  = count_classical_bits(cirq_circuit, fsim_theta, fsim_phi)
    n_data = len(qubit_order)

    qr    = QuantumRegister(24,    'q')
    c_int = ClassicalRegister(n_int, 'c_int')
    c_out = ClassicalRegister(n_data, 'c_out')
    qc    = QuantumCircuit(qr, c_int, c_out)

    idx = 0  # rolling offset into c_int

    for moment in cirq_circuit:
        for op in moment.operations:
            g    = op.gate
            grid = [(q.row, q.col) for q in op.qubits]

            if _is_sqrt_X(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                MBQCTranslatedGates.gate_sqrt_X(qc, a, d, c_int, start_idx=idx)
                idx += MBQCTranslatedGates.N_BITS['sqrt_X']

            elif _is_sqrt_Y(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                MBQCTranslatedGates.gate_sqrt_Y(qc, a, d, c_int, start_idx=idx)
                idx += MBQCTranslatedGates.N_BITS['sqrt_Y']

            elif _is_sqrt_W(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                MBQCTranslatedGates.gate_sqrt_W(qc, a, d, c_int, start_idx=idx)
                idx += MBQCTranslatedGates.N_BITS['sqrt_W']

            elif isinstance(g, cirq.ZPowGate):
                continue   # Rz wrappers dropped

            elif isinstance(g, cirq.FSimGate):
                d1, d2, anc = grid_edge_to_qiskit_indices(grid[0], grid[1])
                MBQCTranslatedGates.gate_two_qubit(
                    qc, d1, anc, d2, c_int,
                    theta=fsim_theta, phi=fsim_phi,
                    Z1=0.0, Z2=0.0, Z3=0.0, Z4=0.0,
                    start_idx=idx,
                )
                idx += MBQCTranslatedGates.n_bits_two_qubit(fsim_theta, fsim_phi)

            else:
                raise ValueError(f'Unhandled gate: {g}')

        qc.barrier()

    for i, q in enumerate(qubit_order):
        d = grid_to_qiskit_index(q.row, q.col)
        qc.measure(d, c_out[i])

    return qc


qc_mbqc = build_mbqc_tetron_circuit(QUBIT_ORDER, CIRCUIT)
print(f'MBQC circuit: {qc_mbqc.num_qubits} qubits, '
      f'{qc_mbqc.num_clbits} clbits, depth = {qc_mbqc.depth()}')


MBQC circuit: 24 qubits, 2527 clbits, depth = 2381


## 4. Build the direct gate-based 12-qubit reference circuit

In [5]:
def build_direct_circuit(qubit_order, cirq_circuit,
                         fsim_theta=FSIM_THETA, fsim_phi=FSIM_PHI):
    """Plain 12-qubit Qiskit circuit using sqrt(X), sqrt(Y), sqrt(W),
    fSim(pi/2, pi/6). Rz wrappers are dropped to match the MBQC translation."""
    n         = len(qubit_order)
    qubit_map = {q: i for i, q in enumerate(qubit_order)}
    qc        = QuantumCircuit(n, n)

    sqW  = UnitaryGate(sqrt_W_matrix(),                   label='√W')
    fsim = UnitaryGate(fSim_matrix(fsim_theta, fsim_phi), label='fSim')

    for moment in cirq_circuit:
        for op in moment.operations:
            g     = op.gate
            q_idx = [qubit_map[q] for q in op.qubits]

            if   _is_sqrt_X(g):                  qc.sx(q_idx[0])
            elif _is_sqrt_Y(g):                  qc.ry(np.pi / 2, q_idx[0])
            elif _is_sqrt_W(g):                  qc.append(sqW,  [q_idx[0]])
            elif isinstance(g, cirq.ZPowGate):   continue                     # Rz dropped
            elif isinstance(g, cirq.FSimGate):   qc.append(fsim, q_idx)
            else: raise ValueError(f'Unhandled gate: {g}')

        qc.barrier()

    for i in range(n):
        qc.measure(i, i)
    return qc


qc_direct = build_direct_circuit(QUBIT_ORDER, CIRCUIT)
print(f'Direct circuit: {qc_direct.num_qubits} qubits, '
      f'{qc_direct.num_clbits} clbits, depth = {qc_direct.depth()}')


Direct circuit: 12 qubits, 12 clbits, depth = 30


## 5. Run both circuits with the noiseless `SamplerV2`

Both circuits use `SHOTS = 4000`. Noiseless simulation.

> **Note on runtime.** The MBQC circuit has ~2500 mid-circuit measurements and
> Pauli feed-forward conditionals, so sampling it with the Aer
> `statevector` backend at 4000 shots can take several minutes. The direct
> 12-qubit circuit is essentially instantaneous. Set `SHOTS_MBQC` smaller if
> you just want a quick smoke test.


In [6]:
import time, os
from concurrent.futures import ThreadPoolExecutor

SHOTS      = 4000   # total shots for both circuits
SHOTS_MBQC = 200  # reduce to e.g. 200 for a quick smoke test
N_CPUS     = os.cpu_count()
print(N_CPUS)
mbqc_t   = transpile(qc_mbqc,   AerSimulator(), optimization_level=0)
direct_t = transpile(qc_direct, AerSimulator(), optimization_level=0)

# ── 1-shot benchmark to project total runtime ─────────────────────────────────
print("Benchmarking 1 MBQC shot …")
t0 = time.perf_counter()
AerSampler(default_shots=1).run([mbqc_t], shots=1).result()
_sec_per_shot = time.perf_counter() - t0
print(f"  1 shot : {_sec_per_shot:.2f} s  →  "
      f"{SHOTS_MBQC} shots ≈ {_sec_per_shot * SHOTS_MBQC / 60:.1f} min single-threaded")


20
Benchmarking 1 MBQC shot …
  1 shot : 276.75 s  →  200 shots ≈ 922.5 min single-threaded


In [ ]:

# ── MBQC: parallel via ThreadPoolExecutor ─────────────────────────────────────
# Aer's C++ statevector core releases the GIL → threads give true CPU parallelism.
_n_workers   = min(N_CPUS, SHOTS_MBQC)
_chunk       = max(1, SHOTS_MBQC // _n_workers)
_shot_chunks = [_chunk] * (_n_workers - 1) + [SHOTS_MBQC - _chunk * (_n_workers - 1)]

def _run_chunk(n):
    return AerSampler(default_shots=n).run([mbqc_t], shots=n).result()[0].data.c_out.get_counts()

print(f"\nSampling MBQC ({SHOTS_MBQC} shots across {_n_workers} threads) …")
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=_n_workers) as pool:
    _chunk_counts = list(pool.map(_run_chunk, _shot_chunks))
elapsed = time.perf_counter() - t0
print(f"  done in {elapsed:.1f} s  ({elapsed / SHOTS_MBQC:.3f} s/shot,  "
      f"~{_sec_per_shot / max(elapsed / SHOTS_MBQC, 1e-9):.1f}x speedup)")

counts_mbqc_raw = Counter()
for _cc in _chunk_counts:
    for bs, n in _cc.items():
        counts_mbqc_raw[bs] += n


In [ ]:

# ── Direct circuit (fast — no need to parallelise) ────────────────────────────
print(f"\nSampling direct circuit ({SHOTS} shots) …")
t0 = time.perf_counter()
res_direct        = AerSampler(default_shots=SHOTS).run([direct_t], shots=SHOTS).result()[0]
counts_direct_raw = res_direct.data.c.get_counts()
print(f"  done in {time.perf_counter() - t0:.1f} s")

# ── Convert to Google-order bitstrings ────────────────────────────────────────
def to_google_order(qiskit_bits):
    """Reverse Qiskit's left-most-is-highest-index ordering so that
    bit i of the returned string corresponds to qubit_order[i]."""
    return qiskit_bits[::-1]

counts_mbqc = Counter()
for bs, n in counts_mbqc_raw.items():
    counts_mbqc[to_google_order(bs)] += n

counts_direct = Counter()
for bs, n in counts_direct_raw.items():
    counts_direct[to_google_order(bs)] += n

print(f"\nMBQC   unique bitstrings: {len(counts_mbqc)}")
print(f"Direct unique bitstrings: {len(counts_direct)}")


## 6. Compare the two output distributions

We plot the **top N** most-probable bitstrings (union over both circuits) on a
shared axis. Bit 0 (leftmost) is qubit (3,3), bit 11 (rightmost) is (5,6).


In [ ]:
TOP_N = 40

shots_m = sum(counts_mbqc.values())
shots_d = sum(counts_direct.values())

union_keys = set(counts_mbqc) | set(counts_direct)
ranked     = sorted(
    union_keys,
    key=lambda k: counts_mbqc.get(k, 0) / shots_m
                 + counts_direct.get(k, 0) / shots_d,
    reverse=True,
)
top_keys = ranked[:TOP_N]

p_mbqc   = [counts_mbqc.get(k,   0) / shots_m for k in top_keys]
p_direct = [counts_direct.get(k, 0) / shots_d for k in top_keys]

x, w = np.arange(len(top_keys)), 0.4
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - w/2, p_mbqc,   w, label=f'MBQC (tetron, {shots_m} shots)',
       color='steelblue', alpha=0.85)
ax.bar(x + w/2, p_direct, w, label=f'Direct  (12 qubit, {shots_d} shots)',
       color='coral',     alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top_keys, rotation=90, fontsize=7, family='monospace')
ax.set_xlabel('12-qubit bitstring  (Google order: bit 0 = qubit (3,3))')
ax.set_ylabel('Probability')
ax.set_title(
    f'Top {TOP_N} bitstrings (of {len(union_keys)} observed) - '
    'fSim(pi/2, pi/6), Rz dropped'
)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Total Variation Distance over the full 12-qubit distribution
tvd = 0.5 * sum(
    abs(counts_mbqc.get(k, 0) / shots_m - counts_direct.get(k, 0) / shots_d)
    for k in union_keys
)
print(f'TVD(MBQC, Direct) = {tvd:.4f}')
print('TVD ~ 1/sqrt(shots) is the expected shot-noise level if the two '
      'implementations agree.')


## 7. Per-qubit marginals

Sanity check: probability that each individual logical qubit is measured `1`,
plotted in `QUBIT_ORDER`.


In [ ]:
def marginals(counts, n=12):
    """P(qubit_i = 1) for i = 0..n-1, given a Google-ordered counts dict."""
    total = sum(counts.values())
    p1    = np.zeros(n)
    for bs, c in counts.items():
        for i, ch in enumerate(bs):
            if ch == '1':
                p1[i] += c
    return p1 / total

m_mbqc   = marginals(counts_mbqc)
m_direct = marginals(counts_direct)
labels   = [f'({q.row},{q.col})' for q in QUBIT_ORDER]

fig, ax = plt.subplots(figsize=(10, 4))
x, w = np.arange(12), 0.4
ax.bar(x - w/2, m_mbqc,   w, label='MBQC',   color='steelblue', alpha=0.85)
ax.bar(x + w/2, m_direct, w, label='Direct', color='coral',     alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45)
ax.set_ylim(0, 1.0)
ax.set_xlabel('Cirq qubit')
ax.set_ylabel('P(qubit = 1)')
ax.set_title('Per-qubit marginal: MBQC vs Direct')
ax.legend()
plt.tight_layout()
plt.show()


## Notes

- **Why agreement is expected.** Both circuits apply the same gate set
  (`sqrt(X)`, `sqrt(Y)`, `sqrt(W)`, `fSim(pi/2, pi/6)`, no `Rz`) to the same
  logical qubits in the same order. The MBQC gadgets implement these unitaries
  exactly via measurement + Pauli feed-forward, so the noiseless TVD should
  shrink as `1/sqrt(shots)`.
- **Bit-ordering reminder.** Inside the Qiskit bitstring, the leftmost
  character is the highest-index classical bit. Both `c_out[i]` and `c[i]`
  index into `QUBIT_ORDER[i]`, so reversing the Qiskit string yields Google
  order: `bs[0]` = qubit (3,3), `bs[11]` = qubit (5,6).
